# FASDD Training Phase 02 — Resume từ Phase 01

Phase 01 bị ngắt do Kaggle timeout. Notebook này resume từ `last.pt`.

**Input cần add vào notebook:**
1. `FASDD_CV COCO Split` — dataset gốc (annotations/ + images/)
2. `FASDD_training_result_phase01` — output phase 01 (chứa last.pt)

In [ ]:
# ==================== SETUP ====================
import os, random, math, time, warnings, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

warnings.filterwarnings('ignore')

for p in Path('/kaggle/input').glob('*'):
    print(p)

# ============ PATHS ============
COCO_ROOT = Path('/kaggle/input/datasets/yuulind/fasdd-cv-coco')
PREV_OUTPUT = Path('/kaggle/input/datasets/dokhang307/fadd-training-batch32-phase02-output')

LAST_PT = PREV_OUTPUT / 'runs/fasdd_train/weights/last.pt'
PREV_BEST = PREV_OUTPUT / 'runs/fasdd_train/weights/best.pt'
PREV_ARGS = PREV_OUTPUT / 'runs/fasdd_train/args.yaml'

WORK = Path('/kaggle/working')
YOLO_DATASET = WORK / 'fasdd_yolo'
RUNS = WORK / 'runs'
SEED = 42
random.seed(SEED); np.random.seed(SEED)

CLASS_NAMES = {0: 'fire', 1: 'smoke'}
SPLITS = ('train', 'val', 'test')
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
TARGET_MAP50 = 0.80
TARGET_FPS = 30
INFER_IMGSZ = 640

# Verify
for f, name in [(LAST_PT, 'last.pt'), (PREV_BEST, 'best.pt')]:
    if f.exists():
        print(f'✅ {name}: {f} ({f.stat().st_size/1e6:.1f} MB)')
    else:
        print(f'❌ {name}: NOT FOUND at {f}')

if PREV_ARGS.exists():
    print(f'\nPhase 01 args:')
    print(open(PREV_ARGS).read()[:500])

In [ ]:
# ==================== INSTALL ====================
!pip install ultralytics opencv-python-headless wandb pycocotools -q
from ultralytics import YOLO
import cv2
import torch
import wandb

print(f'Ultralytics: {__import__("ultralytics").__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# ==================== CONVERT COCO → YOLO (same as phase01) ====================
from pycocotools.coco import COCO

def convert_coco_to_yolo(coco_json, img_src_dir, img_dst_dir, lbl_dst_dir):
    img_dst_dir.mkdir(parents=True, exist_ok=True)
    lbl_dst_dir.mkdir(parents=True, exist_ok=True)

    coco = COCO(str(coco_json))
    cats = coco.loadCats(coco.getCatIds())
    cat_names = {c['id']: c['name'].lower() for c in cats}
    print(f'  Categories: {cat_names}')

    class_map = {}
    for cid, name in cat_names.items():
        if 'fire' in name or 'flame' in name: class_map[cid] = 0
        elif 'smoke' in name: class_map[cid] = 1

    stats = {'images': 0, 'fire': 0, 'smoke': 0, 'negative': 0}
    for img_id in coco.getImgIds():
        img_info = coco.loadImgs(img_id)[0]
        img_w, img_h = img_info['width'], img_info['height']
        fname = img_info['file_name']

        src = img_src_dir / fname
        if not src.exists(): src = img_src_dir / Path(fname).name
        if not src.exists(): continue

        dst_img = img_dst_dir / Path(fname).name
        if not dst_img.exists():
            os.symlink(src.resolve(), dst_img)

        anns = coco.loadAnns(coco.getAnnIds(imgIds=img_id))
        lines = []
        for ann in anns:
            cid = ann['category_id']
            if cid not in class_map: continue
            cls = class_map[cid]
            x, y, w, h = ann['bbox']
            xc = max(0, min(1, (x + w/2) / img_w))
            yc = max(0, min(1, (y + h/2) / img_h))
            wn = max(0, min(1, w / img_w))
            hn = max(0, min(1, h / img_h))
            if wn <= 0 or hn <= 0: continue
            lines.append(f'{cls} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}')
            if cls == 0: stats['fire'] += 1
            else: stats['smoke'] += 1

        (lbl_dst_dir / f'{Path(fname).stem}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''))
        stats['images'] += 1
        if not lines: stats['negative'] += 1
    return stats

print('Converting COCO → YOLO...\n')
for split in SPLITS:
    ann = COCO_ROOT / 'annotations' / f'{split}.json'
    if not ann.exists(): print(f'[{split}] skip'); continue
    print(f'[{split}]')
    s = convert_coco_to_yolo(ann, COCO_ROOT/'images'/split,
                              YOLO_DATASET/'images'/split, YOLO_DATASET/'labels'/split)
    print(f'  {s}\n')

FIXED_YAML = YOLO_DATASET / 'dataset.yaml'
FIXED_YAML.write_text(f"""path: {YOLO_DATASET}
train: images/train
val: images/val
test: images/test

nc: 2
names:
  0: fire
  1: smoke
""")
print(f'✅ {FIXED_YAML}')

In [ ]:
# ==================== W&B ====================
from datetime import datetime

USE_WANDB = True
try:
    from kaggle_secrets import UserSecretsClient
    WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
    print('✅ W&B key from Kaggle Secrets')
except:
    WANDB_API_KEY = "PASTE_KEY_HERE"
    print('⚠️  Hardcoded key — dùng Kaggle Secrets!')

if USE_WANDB:
    try:
        os.environ["WANDB_API_KEY"] = WANDB_API_KEY
        os.environ["WANDB_PROJECT"] = "wildfire-spread-prediction"
        wandb.login(key=WANDB_API_KEY, relogin=True)
        run = wandb.init(
            project="wildfire-spread-prediction",
            name=f"fasdd_phase02_{datetime.now().strftime('%Y%m%d_%H%M')}",
            config={"phase": "02_resume", "dataset": "FASDD_CV_COCO",
                    "resume_from": str(LAST_PT), "imgsz": 960, "epochs": 150},
            tags=["yolov8s", "fasdd", "phase02", "resume"],
        )
        print(f'✅ W&B: {run.url}')
    except Exception as e:
        print(f'❌ W&B failed: {e}'); USE_WANDB = False

## Resume Training

**Cách resume đúng trong Ultralytics:**

`resume=True` + `model=last.pt` → Ultralytics tự khôi phục epoch, optimizer state, lr schedule.

**Lưu ý quan trọng:** Ultralytics khi resume sẽ ghi output vào **cùng thư mục gốc** được lưu trong last.pt.
Nhưng thư mục gốc nằm trong `/kaggle/input/` (read-only) → cần copy last.pt ra working trước.

In [ ]:
# ==================== COPY LAST.PT + RESUME ====================
# Ultralytics resume cần ghi vào thư mục gốc của run.
# last.pt nằm trong input (read-only) → copy ra working rồi sửa save_dir.

# Copy toàn bộ run folder ra working
RESUME_DIR = RUNS / 'fasdd_train'
if not RESUME_DIR.exists():
    shutil.copytree(PREV_OUTPUT / 'runs/fasdd_train', RESUME_DIR)
    print(f'✅ Copied run folder to {RESUME_DIR}')
else:
    print(f'Run folder already exists at {RESUME_DIR}')

RESUME_LAST = RESUME_DIR / 'weights' / 'last.pt'
assert RESUME_LAST.exists(), f'last.pt not found at {RESUME_LAST}'
print(f'Resume from: {RESUME_LAST}')

# Load model từ last.pt đã copy
model = YOLO(str(RESUME_LAST))

# Resume training — Ultralytics tự biết epoch hiện tại từ checkpoint
model.train(
    resume=True,
    data=str(FIXED_YAML),
    project=str(RUNS),
    name='fasdd_train',
    exist_ok=True,
)

# Tìm best.pt mới
BEST = RESUME_DIR / 'weights' / 'best.pt'
if BEST.exists():
    print(f'\n✅ Best model: {BEST} ({BEST.stat().st_size/1e6:.1f} MB)')
else:
    print(f'⚠️  best.pt không tìm thấy tại {BEST}')
    # Fallback
    BEST = PREV_BEST
    print(f'   Dùng best.pt từ phase01: {BEST}')

## Evaluation + FPS + Direction + Spread

In [ ]:
# ==================== EVALUATE ====================
model = YOLO(str(BEST))

val_metrics = model.val(data=str(FIXED_YAML), split='val', imgsz=960, batch=8,
                        plots=True, project=str(RUNS), name='eval_val', exist_ok=True)
test_metrics = model.val(data=str(FIXED_YAML), split='test', imgsz=960, batch=8,
                         plots=True, project=str(RUNS), name='eval_test', exist_ok=True)
test_640 = model.val(data=str(FIXED_YAML), split='test', imgsz=640, batch=16,
                     plots=False, project=str(RUNS), name='eval_640', exist_ok=True)

print('\nResults')
print('=' * 65)
print(f'{"Split":<8} {"imgsz":>6} {"P":>8} {"R":>8} {"mAP50":>8} {"mAP50-95":>10}')
print('-' * 65)
for label, m, sz in [('val', val_metrics, 960), ('test', test_metrics, 960), ('test', test_640, 640)]:
    mp, mr, map50, map5095 = m.box.mean_results()
    s = '✅' if map50 >= TARGET_MAP50 else '❌'
    print(f'{label:<8} {sz:>6} {mp:>8.3f} {mr:>8.3f} {map50:>8.3f} {map5095:>10.3f} {s}')
print('-' * 65)

print('\nPer-class (test 960):')
for i, name in test_metrics.names.items():
    p = test_metrics.box.class_result(i)
    print(f'  {name:<10} P={p[0]:.3f} R={p[1]:.3f} mAP50={p[2]:.3f}')

mp_t, mr_t, map50_t, map5095_t = test_metrics.box.mean_results()
_, _, map50_640, _ = test_640.box.mean_results()
smoke_map50 = test_metrics.box.class_result(1)[2]
fire_map50 = test_metrics.box.class_result(0)[2]

In [ ]:
# ==================== FPS ====================
model = YOLO(str(BEST))
test_dir = YOLO_DATASET / 'images' / 'test'
test_imgs = sorted([str(p) for p in test_dir.iterdir() if p.suffix.lower() in IMG_EXTS])

for i in range(50):
    _ = model.predict(test_imgs[i % len(test_imgs)], imgsz=INFER_IMGSZ, conf=0.25, verbose=False)

times = []
for i in range(200):
    t0 = time.perf_counter()
    _ = model.predict(test_imgs[i % len(test_imgs)], imgsz=INFER_IMGSZ, conf=0.25, verbose=False)
    times.append(time.perf_counter() - t0)

times = np.array(times)
fps_mean = 1.0 / times.mean()
fps_median = 1.0 / np.median(times)
latency_mean = times.mean() * 1000

print(f'FPS: mean={fps_mean:.1f} median={fps_median:.1f} latency={latency_mean:.1f}ms')
print(f'Status: {"✅" if fps_mean >= TARGET_FPS else "❌"} (target: {TARGET_FPS})')

In [ ]:
# ==================== DIRECTION + SPREAD ====================
def direction_from_geometry(boxes, img=None, img_w=640, img_h=640):
    fires = [b for b in boxes if b[0] == 0]
    smokes = [b for b in boxes if b[0] == 1]
    ra, rb = None, None
    if fires and smokes:
        f = max(fires, key=lambda b: b[3]*b[4])
        s = max(smokes, key=lambda b: b[3]*b[4])
        dx, dy = s[1]-f[1], -(s[2]-f[2])
        dist = math.hypot(dx, dy)
        if dist > 0.02:
            ra = {'theta': math.degrees(math.atan2(dy, dx)) % 360, 'conf': min(dist*5, 1.0)}
    if smokes and img is not None:
        s = max(smokes, key=lambda b: b[3]*b[4])
        _, xc, yc, w, h = s
        x1, y1 = int((xc-w/2)*img_w), int((yc-h/2)*img_h)
        x2, y2 = int((xc+w/2)*img_w), int((yc+h/2)*img_h)
        crop = img[max(y1,0):y2, max(x1,0):x2]
        if crop.size > 0:
            hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
            ys, xs = np.nonzero((hsv[:,:,1]<60)&(hsv[:,:,2]>90))
            if len(xs) > 200:
                pts = np.stack([xs-xs.mean(), ys-ys.mean()])
                eigval, eigvec = np.linalg.eigh(np.cov(pts))
                v = eigvec[:,-1]; elong = eigval[-1]/max(eigval[0],1e-6)
                if elong > 1.5:
                    if v[1] > 0: v = -v
                    rb = {'theta': math.degrees(math.atan2(-v[1],v[0]))%360, 'conf': min((elong-1)/4,1.0)}
    if ra and rb:
        diff = abs((ra['theta']-rb['theta']+180)%360-180)
        best = ra if ra['conf']>=rb['conf'] else rb
        if diff <= 60: return {'theta': best['theta'], 'conf': (ra['conf']+rb['conf'])/2, 'method': 'A+B'}
        return {'theta': best['theta'], 'conf': best['conf']*0.3, 'method': 'A+B_conflict'}
    if ra: return {'theta': ra['theta'], 'conf': ra['conf']*0.8, 'method': 'A'}
    if rb: return {'theta': rb['theta'], 'conf': rb['conf']*0.7, 'method': 'B'}
    return None

def make_spread_ellipse(fc_px, wind_deg, w, h, r=0.15, e=0.8):
    from matplotlib.patches import Ellipse
    a = r*max(w,h); b = a*(1-e)
    ox = a*0.4*math.cos(math.radians(wind_deg)); oy = -a*0.4*math.sin(math.radians(wind_deg))
    return Ellipse((fc_px[0]+ox,fc_px[1]+oy), 2*a, 2*b, angle=-wind_deg,
                   fc='orange', alpha=0.25, ec='orange', lw=2, ls='--')

# Run on test
model = YOLO(str(BEST))
test_imgs_list = sorted([p for p in (YOLO_DATASET/'images'/'test').iterdir() if p.suffix.lower() in IMG_EXTS])

directions = []
for img_path in test_imgs_list:
    res = model.predict(str(img_path), imgsz=INFER_IMGSZ, conf=0.25, verbose=False)
    img = cv2.imread(str(img_path)); h_, w_ = img.shape[:2]
    boxes = []
    for box in res[0].boxes:
        c = int(box.cls[0]); x1,y1,x2,y2 = box.xyxy[0].cpu().numpy()
        boxes.append((c,(x1+x2)/2/w_,(y1+y2)/2/h_,(x2-x1)/w_,(y2-y1)/h_))
    d = direction_from_geometry(boxes, img, w_, h_)
    if d: d['image'] = img_path.stem; directions.append(d)

dir_df = pd.DataFrame(directions)
print(f'Directions: {len(dir_df)}/{len(test_imgs_list)} ({len(dir_df)/max(len(test_imgs_list),1)*100:.0f}%)')
if len(dir_df):
    print(f'Methods: {dir_df["method"].value_counts().to_dict()}')
    print(f'Mean conf: {dir_df["conf"].mean():.2f}')

In [ ]:
# ==================== SPREAD FIGURE ====================
if len(dir_df) > 0:
    top = dir_df.nlargest(6, 'conf')
    n = min(6, len(top)); cols = 3; rows = (n+cols-1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
    axes_flat = np.array(axes).flat if n > 1 else [axes]

    for ax, (_, row) in zip(axes_flat, top.iterrows()):
        for ext in IMG_EXTS:
            p = YOLO_DATASET/'images'/'test'/f"{row['image']}{ext}"
            if p.exists():
                img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
                results = model.predict(str(p), imgsz=INFER_IMGSZ, conf=0.25, verbose=False)
                h_, w_ = img.shape[:2]
                boxes = []
                for box in results[0].boxes:
                    c = int(box.cls[0]); x1,y1,x2,y2 = box.xyxy[0].cpu().numpy()
                    boxes.append((c,(x1+x2)/2/w_,(y1+y2)/2/h_,(x2-x1)/w_,(y2-y1)/h_))
                break
        else: continue

        ax.imshow(img)
        for box in results[0].boxes:
            cls = int(box.cls[0]); x1,y1,x2,y2 = map(int, box.xyxy[0])
            ax.add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1, fill=False,
                                            edgecolor='red' if cls==0 else 'deepskyblue', lw=2))
        fire_boxes = [b for b in boxes if b[0]==0]
        if fire_boxes:
            fb = fire_boxes[0]; fcx, fcy = fb[1]*w_, fb[2]*h_
            alen = min(w_,h_)*0.2
            dx = alen*math.cos(math.radians(row['theta']))
            dy = -alen*math.sin(math.radians(row['theta']))
            ax.annotate('', xy=(fcx+dx,fcy+dy), xytext=(fcx,fcy),
                        arrowprops=dict(arrowstyle='->', color='lime', lw=3))
            ax.add_patch(make_spread_ellipse((fcx,fcy), row['theta'], w_, h_))
        ax.set_title(f"wind={row['theta']:.0f}° conf={row['conf']:.2f} [{row['method']}]", fontsize=9)
        ax.axis('off')

    for ax in list(axes_flat)[n:]: ax.set_visible(False)
    plt.suptitle('Spread Prediction', fontsize=10)
    plt.tight_layout()
    plt.savefig(WORK/'fig_spread.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# ==================== SUMMARY + W&B ====================
print('=' * 70)
print('NGHIỆM THU')
print('=' * 70)

checks = [
    ('mAP@50 (test, 960px)', map50_t, TARGET_MAP50, map50_t >= TARGET_MAP50),
    ('mAP@50 (test, 640px)', map50_640, TARGET_MAP50, map50_640 >= TARGET_MAP50),
    ('FPS (640px)', fps_mean, TARGET_FPS, fps_mean >= TARGET_FPS),
    ('Smoke mAP@50', smoke_map50, 0.50, smoke_map50 >= 0.50),
]
all_pass = True
for name, val, target, passed in checks:
    s = '✅ PASS' if passed else '❌ FAIL'
    print(f'  {name:<30} {val:>8.3f}  (target: {target})  {s}')
    if not passed: all_pass = False

if len(dir_df):
    print(f'\n  Directions: {len(dir_df)}/{len(test_imgs_list)} = {len(dir_df)/max(len(test_imgs_list),1)*100:.0f}%')

if USE_WANDB:
    try:
        wandb.log({
            "test/mAP50_960": float(map50_t), "test/mAP50_640": float(map50_640),
            "test/mAP50-95": float(map5095_t), "test/precision": float(mp_t),
            "test/recall": float(mr_t), "test/fire_mAP50": float(fire_map50),
            "test/smoke_mAP50": float(smoke_map50),
            "benchmark/fps_mean": float(fps_mean), "benchmark/latency_ms": float(latency_mean),
        })
        if len(dir_df):
            wandb.log({"direction/count": len(dir_df), "direction/mean_conf": float(dir_df['conf'].mean())})
        for f in sorted(WORK.glob('fig_*.png')):
            wandb.log({f"figures/{f.stem}": wandb.Image(str(f))})
        if BEST.exists():
            art = wandb.Artifact("yolov8s-fasdd-phase02", type="model")
            art.add_file(str(BEST), name="best.pt")
            wandb.log_artifact(art)
        wandb.summary.update({"final/mAP50": float(map50_t), "final/smoke_mAP50": float(smoke_map50),
                               "final/fps": float(fps_mean), "final/all_pass": all_pass})
        wandb.finish()
        print(f'\n✅ W&B: {run.url}')
    except Exception as e:
        print(f'W&B: {e}')

if len(dir_df): dir_df.to_csv(WORK/'smoke_directions.csv', index=False)
print('=' * 70)